# Recomendation Engine

Technologies: XGBoost and rule Engine

## Recomendation Types

Spending Warning: User compared to themself and other uers

Savings Suggestion: Estimated monthly savings potential

Budget Advice: Top overspent category compared to others

Behavioral Nudge: Velocity, weekrnd patterns, merchant diversity

### Comparison Types
1. Personal - user compared to their own history
2. Peer - user compared to other users

In [1]:
import pandas as pd
import numpy as np

In [6]:
fm_encoded      = pd.read_csv('../features/fm_encoded.csv', index_col='user_id')
transactions_df = pd.read_csv('../features/transactions_enriched.csv', parse_dates=['date'])
anomaly_df      = pd.read_csv('outputs/anomaly_scores.csv', index_col='user_id')

print(f'fm_encoded: {fm_encoded.shape[0]} users, {fm_encoded.shape[1]} columns')
print(f'transactions_df: {len(transactions_df):,} rows')
print(f'anomaly_df: {anomaly_df.shape[0]} users')
print(f'\ntask_segment distribution:')
print(fm_encoded['task_segment'].value_counts())

fm_encoded: 388 users, 120 columns
transactions_df: 22,602 rows
anomaly_df: 388 users

task_segment distribution:
task_segment
Single-Tasker          187
Low Activity/Trial      99
Consistent Weekly       64
Healthy Active User     33
High-Intensity User      5
Name: count, dtype: int64


## Create Peer Groups

Peer groups created from task_segment from fm_encoded. For each group calculate median value across all key features to be able compare every recomendation.

In [ ]:
PEER_BENCHMARK_COLS = [
    'vel_7d', 'vel_30d', 'avg_amt_30d', 'count_30d',
    'accounts.MEAN(transactions.amount)',
    'accounts.MAX(transactions.amount)',
    'accounts.MEAN(monthly_stats.total_spend)',
    'accounts.MAX(monthly_stats.total_spend)',
    'accounts.SUM(monthly_stats.total_spend)',
    'merchant_diversity', 'new_merchant_count',
    'days_since_last',
]

CATEGORY_COLS = [
    c for c in fm_encoded.columns
    if 'monthly_stats' in c and 'MEAN' in c and 'total' not in c and 'avg_transaction' not in c
]

available_benchmark = [c for c in PEER_BENCHMARK_COLS if c in fm_encoded.columns]
available_categories = [c for c in CATEGORY_COLS if c in fm_encoded.columns]

print(f'Benchmark features:  {len(available_benchmark)}')
print(f'Category features:   {len(available_categories)}')
print(f'{[c.split("monthly_stats.")[1].rstrip(")") for c in available_categories]}')

Benchmark features:  12
Category features:   11
['Fitness', 'Food', 'Friend Activities', 'Gifts', 'Hobbies', 'Housing and Utilities', 'Medical/Dental', 'Personal Hygiene', 'Subscriptions', 'Transportation', 'Travel']


In [9]:
peer_benchmarks = (
    fm_encoded.groupby('task_segment')[available_benchmark + available_categories].median()
)

print(f'\nPeer benchmarks computed for {len(peer_benchmarks)} segments:')
print(peer_benchmarks[available_benchmark].round(2))


Peer benchmarks computed for 5 segments:
                     vel_7d  vel_30d  avg_amt_30d  count_30d  \
task_segment                                                   
Consistent Weekly       2.0      3.0       281.26        3.0   
Healthy Active User     3.0      6.0       632.96        6.0   
High-Intensity User    22.0     66.0        -0.60       66.0   
Low Activity/Trial      0.0      0.0         0.00        0.0   
Single-Tasker           1.0      2.0       200.00        2.0   

                     accounts.MEAN(transactions.amount)  \
task_segment                                              
Consistent Weekly                               1312.37   
Healthy Active User                              692.83   
High-Intensity User                                0.01   
Low Activity/Trial                                 0.00   
Single-Tasker                                   1000.00   

                     accounts.MAX(transactions.amount)  \
task_segment                         